# Research Portfolio FP&A & Grant Performance Engine
### Multi-Year Grant Forecasting, FTE Labor Recovery & Executive Variance Analysis

This financial planning engine models multi-year research grant accounting, project-level labor cost recovery, operational overhead absorption, and budget-vs-actual variance decomposition across scientific research programs. It provides an automated decision-support pipeline designed to bridge financial modeling with operational business partnering for non-financial stakeholders.

In [7]:
# 1. Imports & Environment Configuration
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, date

# Display parameters
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment configured. Python version:", sys.version.split()[0])

Environment configured. Python version: 3.12.13


In [8]:
# 2. Synthetic Multi-Year Grant Portfolio & GL Ledger Generator

def generate_research_portfolio_data(seed: int = 101) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Generates a realistic multi-year research funding portfolio and 12-month
    General Ledger actuals vs. budget records across 4 scientific divisions.
    """
    np.random.seed(seed)

    divisions = [
        "Applied Agricultural Science",
        "Ecosystem & Environmental Risk",
        "Biotechnology & Health Solutions",
        "Sustainable Materials & Forestry"
    ]

    funding_mechanisms = [
        "Government Crown Core",
        "Competitive Research Grant",
        "Commercial Enterprise Contract",
        "International Research Consortium"
    ]

    # 1. Master Project Register
    projects = []
    p_counter = 101

    for div in divisions:
        for i in range(1, 6):  # 5 major programmes per division
            p_id = f"RND-{p_counter}"
            p_counter += 1
            f_stream = np.random.choice(funding_mechanisms, p=[0.40, 0.35, 0.15, 0.10])
            total_funding = np.random.uniform(1_500_000, 5_000_000)
            duration_months = np.random.choice([24, 36, 48])
            planned_fte = np.random.uniform(2.0, 7.5)
            chargeout_rate = np.random.uniform(140.0, 190.0)  # $ / hour

            projects.append({
                "Project_ID": p_id,
                "Division": div,
                "Project_Name": f"{div} Programme {i}",
                "Funding_Mechanism": f_stream,
                "Total_Funding_NZD": total_funding,
                "Duration_Months": duration_months,
                "Planned_FTE": planned_fte,
                "Hourly_Chargeout_Rate": chargeout_rate
            })

    df_projects = pd.DataFrame(projects)

    # 2. Monthly Financial Records (12-Month Financial Year)
    months = pd.date_range(start="2025-07-01", periods=12, freq="MS")
    gl_records = []

    for _, prj in df_projects.iterrows():
        base_monthly_revenue = prj["Total_Funding_NZD"] / prj["Duration_Months"]
        base_fte_hours = prj["Planned_FTE"] * 140  # 140 productive hours/month per FTE
        base_direct_labor_cost = base_fte_hours * 72.50  # Base cost $72.50/hr
        base_fte_recovery = base_fte_hours * prj["Hourly_Chargeout_Rate"]
        base_lab_opex = base_monthly_revenue * np.random.uniform(0.14, 0.22)
        base_overhead = base_monthly_revenue * 0.18  # 18% Institutional Indirect Cost
        base_depreciation = np.random.uniform(5_000, 15_000)

        for m in months:
            # Operational volatility factors
            rev_var_factor = np.random.normal(loc=0.98, scale=0.07)
            fte_var_factor = np.random.normal(loc=0.96, scale=0.05)
            lab_var_factor = np.random.normal(loc=1.04, scale=0.10)

            # Budget values
            b_rev = base_monthly_revenue
            b_hours = base_fte_hours
            b_recovery = base_fte_recovery
            b_labor_cost = base_direct_labor_cost
            b_lab_opex = base_lab_opex
            b_overhead = base_overhead
            b_deprec = base_depreciation

            # Actual values
            a_rev = b_rev * rev_var_factor
            a_hours = b_hours * fte_var_factor
            a_recovery = a_hours * prj["Hourly_Chargeout_Rate"]
            a_labor_cost = a_hours * 72.50 * np.random.uniform(0.99, 1.04)  # Wage inflation
            a_lab_opex = b_lab_opex * lab_var_factor
            a_overhead = b_overhead
            a_deprec = b_deprec * np.random.choice([1.0, 1.05])

            gl_records.append({
                "Period": m,
                "Project_ID": prj["Project_ID"],
                "Division": prj["Division"],
                "Funding_Mechanism": prj["Funding_Mechanism"],
                "Budget_Revenue": b_rev,
                "Actual_Revenue": a_rev,
                "Budget_FTE_Hours": b_hours,
                "Actual_FTE_Hours": a_hours,
                "Budget_Labor_Recovery": b_recovery,
                "Actual_Labor_Recovery": a_recovery,
                "Budget_Direct_Labor_Cost": b_labor_cost,
                "Actual_Direct_Labor_Cost": a_labor_cost,
                "Budget_Lab_OpEx": b_lab_opex,
                "Actual_Lab_OpEx": a_lab_opex,
                "Budget_Overhead": b_overhead,
                "Actual_Overhead": a_overhead,
                "Budget_Depreciation": b_deprec,
                "Actual_Depreciation": a_deprec
            })

    df_gl = pd.DataFrame(gl_records)
    return df_projects, df_gl

df_projects, df_gl = generate_research_portfolio_data()
print(f"Generated {len(df_projects)} research projects across {df_projects['Division'].nunique()} divisions.")
print(f"Generated {len(df_gl)} GL transaction records spanning 12 operating months.")

Generated 20 research projects across 4 divisions.
Generated 240 GL transaction records spanning 12 operating months.


In [9]:
# 3. Core Financial & Variance Decomposition Engine

class PortfolioFinancialEngine:
    def __init__(self, df_gl: pd.DataFrame, df_projects: pd.DataFrame):
        self.df_gl = df_gl.copy()
        self.df_projects = df_projects.copy()
        self._calculate_financial_metrics()

    def _calculate_financial_metrics(self):
        """Calculates Net Margins, Cost Totals, and Detailed Variances."""
        # Total Operating Costs
        self.df_gl["Budget_Total_Cost"] = (
            self.df_gl["Budget_Direct_Labor_Cost"] +
            self.df_gl["Budget_Lab_OpEx"] +
            self.df_gl["Budget_Overhead"] +
            self.df_gl["Budget_Depreciation"]
        )
        self.df_gl["Actual_Total_Cost"] = (
            self.df_gl["Actual_Direct_Labor_Cost"] +
            self.df_gl["Actual_Lab_OpEx"] +
            self.df_gl["Actual_Overhead"] +
            self.df_gl["Actual_Depreciation"]
        )

        # Net Operating Margins
        self.df_gl["Budget_Net_Margin"] = self.df_gl["Budget_Revenue"] - self.df_gl["Budget_Total_Cost"]
        self.df_gl["Actual_Net_Margin"] = self.df_gl["Actual_Revenue"] - self.df_gl["Actual_Total_Cost"]

        # Primary Variances (Favorable > 0, Unfavorable < 0)
        self.df_gl["Revenue_Variance"] = self.df_gl["Actual_Revenue"] - self.df_gl["Budget_Revenue"]
        self.df_gl["Labor_Cost_Variance"] = self.df_gl["Budget_Direct_Labor_Cost"] - self.df_gl["Actual_Direct_Labor_Cost"]
        self.df_gl["Lab_OpEx_Variance"] = self.df_gl["Budget_Lab_OpEx"] - self.df_gl["Actual_Lab_OpEx"]
        self.df_gl["Depreciation_Variance"] = self.df_gl["Budget_Depreciation"] - self.df_gl["Actual_Depreciation"]
        self.df_gl["Net_Margin_Variance"] = self.df_gl["Actual_Net_Margin"] - self.df_gl["Budget_Net_Margin"]

        # Labor Rate vs. Efficiency Decomposition
        actual_hourly_cost = self.df_gl["Actual_Direct_Labor_Cost"] / np.where(self.df_gl["Actual_FTE_Hours"] == 0, 1, self.df_gl["Actual_FTE_Hours"])
        budget_hourly_cost = self.df_gl["Budget_Direct_Labor_Cost"] / np.where(self.df_gl["Budget_FTE_Hours"] == 0, 1, self.df_gl["Budget_FTE_Hours"])

        self.df_gl["Labor_Rate_Variance"] = (budget_hourly_cost - actual_hourly_cost) * self.df_gl["Actual_FTE_Hours"]
        self.df_gl["Labor_Efficiency_Variance"] = (self.df_gl["Budget_FTE_Hours"] - self.df_gl["Actual_FTE_Hours"]) * budget_hourly_cost

    def get_division_summary(self) -> pd.DataFrame:
        """Returns consolidated metrics aggregated by scientific division."""
        summary = self.df_gl.groupby("Division").agg({
            "Budget_Revenue": "sum",
            "Actual_Revenue": "sum",
            "Revenue_Variance": "sum",
            "Budget_Direct_Labor_Cost": "sum",
            "Actual_Direct_Labor_Cost": "sum",
            "Budget_Net_Margin": "sum",
            "Actual_Net_Margin": "sum",
            "Net_Margin_Variance": "sum",
            "Budget_FTE_Hours": "sum",
            "Actual_FTE_Hours": "sum"
        }).reset_index()

        summary["Revenue_Realization_%"] = (summary["Actual_Revenue"] / summary["Budget_Revenue"]) * 100
        summary["FTE_Utilization_%"] = (summary["Actual_FTE_Hours"] / summary["Budget_FTE_Hours"]) * 100
        summary["Actual_Margin_%"] = (summary["Actual_Net_Margin"] / summary["Actual_Revenue"]) * 100
        return summary

    def get_waterfall_bridge(self) -> dict:
        """Calculates the total institution Net Margin Variance Waterfall elements."""
        return {
            "Budget Net Margin": self.df_gl["Budget_Net_Margin"].sum(),
            "Revenue Realization": self.df_gl["Revenue_Variance"].sum(),
            "Labor Cost Variance": self.df_gl["Labor_Cost_Variance"].sum(),
            "Lab Consumables OpEx": self.df_gl["Lab_OpEx_Variance"].sum(),
            "Depreciation & Fixed": self.df_gl["Depreciation_Variance"].sum(),
            "Actual Net Margin": self.df_gl["Actual_Net_Margin"].sum()
        }

engine = PortfolioFinancialEngine(df_gl, df_projects)
div_summary = engine.get_division_summary()
display(div_summary[["Division", "Budget_Revenue", "Actual_Revenue", "Revenue_Realization_%", "FTE_Utilization_%", "Actual_Margin_%"]])

,Division,Budget_Revenue,Actual_Revenue,Revenue_Realization_%,FTE_Utilization_%,Actual_Margin_%
0,Applied Agricultural Science,6082835.35,5955342.47,97.90,95.67,9.31
1,Biotechnology & Health Solutions,5811319.29,5610719.30,96.55,94.91,6.94
2,Ecosystem & Environmental Risk,6026145.36,5957593.53,98.86,96.72,0.16
3,Sustainable Materials & Forestry,4632383.55,4571467.79,98.69,96.41,-3.94


In [10]:
# 4. Executive Variance Waterfall & KPI Dashboards

def render_executive_charts(engine: PortfolioFinancialEngine):
    """Renders interactive Plotly visual reporting packs."""
    # 1. Waterfall Bridge
    bridge = engine.get_waterfall_bridge()
    categories = list(bridge.keys())
    values = list(bridge.values())

    measures = ["absolute", "relative", "relative", "relative", "relative", "total"]

    fig_waterfall = go.Figure(go.Waterfall(
        name="Net Margin Variance",
        orientation="v",
        measure=measures,
        x=categories,
        textposition="outside",
        text=[f"${v/1e3:.1f}k" for v in values],
        y=values,
        connector={"line": {"color": "#444"}},
        increasing={"marker": {"color": "#2ca02c"}},
        decreasing={"marker": {"color": "#d62728"}},
        totals={"marker": {"color": "#1f77b4"}}
    ))

    fig_waterfall.update_layout(
        title="Annual Net Operating Contribution Variance Bridge ($NZD)",
        height=450,
        yaxis_title="NZD ($)",
        template="plotly_white"
    )
    fig_waterfall.show()

    # 2. Division Comparison Chart
    div_perf = engine.get_division_summary()
    fig_div = make_subplots(rows=1, cols=2, subplot_titles=("Revenue: Budget vs Actual", "Scientific FTE Utilization (%)"))

    fig_div.add_trace(
        go.Bar(name="Budget Revenue", x=div_perf["Division"], y=div_perf["Budget_Revenue"], marker_color="#9ecae1"),
        row=1, col=1
    )
    fig_div.add_trace(
        go.Bar(name="Actual Revenue", x=div_perf["Division"], y=div_perf["Actual_Revenue"], marker_color="#2171b5"),
        row=1, col=1
    )

    fig_div.add_trace(
        go.Bar(
            name="FTE Utilization %",
            x=div_perf["Division"],
            y=div_perf["FTE_Utilization_%"],
            marker_color=np.where(div_perf["FTE_Utilization_%"] >= 95, "#31a354", "#e6550d"),
            text=[f"{v:.1f}%" for v in div_perf["FTE_Utilization_%"]],
            textposition="auto"
        ),
        row=1, col=2
    )

    fig_div.update_layout(height=450, barmode="group", template="plotly_white")
    fig_div.show()

render_executive_charts(engine)

In [11]:
# 5. Automated Executive Advisory Brief Generator

class ExecutiveAdvisoryReporter:
    @staticmethod
    def generate_brief(engine: PortfolioFinancialEngine) -> str:
        div_summary = engine.get_division_summary()
        bridge = engine.get_waterfall_bridge()

        brief = []
        brief.append("=" * 80)
        brief.append("FINANCE BUSINESS PARTNER EXECUTIVE BRIEFING | ANNUAL PORTFOLIO REVIEW")
        brief.append(f"Report Date: {date.today().strftime('%d %B %Y')} | Currency: NZD ($)")
        brief.append("=" * 80)
        brief.append("")

        # 1. Executive Summary
        tot_b_rev = div_summary["Budget_Revenue"].sum()
        tot_a_rev = div_summary["Actual_Revenue"].sum()
        rev_gap = tot_a_rev - tot_b_rev
        direction = "ahead of" if rev_gap >= 0 else "behind"

        brief.append("1. PORTFOLIO FINANCIAL TRAJECTORY")
        brief.append(f"• Total Research Revenue: ${tot_a_rev/1e6:.2f}M vs Budget ${tot_b_rev/1e6:.2f}M ({direction} target by ${abs(rev_gap)/1e3:.1f}k or {(tot_a_rev/tot_b_rev - 1)*100:+.1f}%).")
        brief.append(f"• Net Margin Delivered: ${bridge['Actual Net Margin']/1e3:.1f}k vs Planned ${bridge['Budget Net Margin']/1e3:.1f}k.")
        brief.append("")

        # 2. Division Breakdown & Risk Flags
        brief.append("2. DIVISIONAL PERFORMANCE & OPERATIONAL CAPACITY")
        for _, row in div_summary.iterrows():
            div = row["Division"]
            rev_achieve = row["Revenue_Realization_%"]
            fte_util = row["FTE_Utilization_%"]
            margin = row["Actual_Margin_%"]

            status = "ON TRACK" if (rev_achieve >= 95 and fte_util >= 93) else "ATTENTION REQUIRED"
            brief.append(f"[{status}] - {div}:")
            brief.append(f"  - Revenue Realization: {rev_achieve:.1f}% | Net Margin: {margin:.1f}%")
            brief.append(f"  - FTE Labor Utilization: {fte_util:.1f}% of planned billable capacity.")

            if fte_util < 95:
                unrecovered_hours = row["Budget_FTE_Hours"] - row["Actual_FTE_Hours"]
                unrealized_rec = unrecovered_hours * 165.0
                brief.append(f"  * ADVISORY ACTION: Capacity slippage detected. Estimated ${unrealized_rec/1e3:.1f}k in unrecovered labor.")
                brief.append(f"    Action: Work with Programme Leads to reassign scientific FTEs to contestable grant deliverables.")
            else:
                brief.append("  * ADVISORY ACTION: High operational absorption maintained. Monitor project milestones to avoid delivery bottlenecks.")
            brief.append("")

        # 3. Recommendations
        brief.append("3. STRATEGIC RECOMMENDATIONS FOR OPERATIONAL DIRECTORS")
        brief.append("• Milestone Review: Synchronize quarterly scientific milestones with invoicing schedules to reduce revenue lag.")
        brief.append("• OpEx Procurement: Standardize laboratory consumable vendor agreements to capture volume discount synergies.")
        brief.append("• Chargeout Harmonization: Re-benchmark FTE recovery rates against current wage inflation to prevent margin dilution.")
        brief.append("=" * 80)

        return "\n".join(brief)

advisory_brief = ExecutiveAdvisoryReporter.generate_brief(engine)
print(advisory_brief)

FINANCE BUSINESS PARTNER EXECUTIVE BRIEFING | ANNUAL PORTFOLIO REVIEW
Report Date: 15 August 2026 | Currency: NZD ($)

1. PORTFOLIO FINANCIAL TRAJECTORY
• Total Research Revenue: $22.10M vs Budget $22.55M (behind target by $457.6k or -2.0%).
• Net Margin Delivered: $773.2k vs Planned $1276.5k.

2. DIVISIONAL PERFORMANCE & OPERATIONAL CAPACITY
[ON TRACK] - Applied Agricultural Science:
  - Revenue Realization: 97.9% | Net Margin: 9.3%
  - FTE Labor Utilization: 95.7% of planned billable capacity.
  * ADVISORY ACTION: High operational absorption maintained. Monitor project milestones to avoid delivery bottlenecks.

[ON TRACK] - Biotechnology & Health Solutions:
  - Revenue Realization: 96.5% | Net Margin: 6.9%
  - FTE Labor Utilization: 94.9% of planned billable capacity.
  * ADVISORY ACTION: Capacity slippage detected. Estimated $303.0k in unrecovered labor.
    Action: Work with Programme Leads to reassign scientific FTEs to contestable grant deliverables.

[ON TRACK] - Ecosystem & Env

In [12]:
# 6. Data Export Pipeline (For GitHub & Portfolio Documentation)

# Save datasets and executive text
df_gl.to_csv("research_portfolio_gl_data.csv", index=False)
div_summary.to_csv("division_summary_report.csv", index=False)

with open("executive_advisory_brief.txt", "w") as f:
    f.write(advisory_brief)

print("Files successfully written to disk:")
print("1. research_portfolio_gl_data.csv")
print("2. division_summary_report.csv")
print("3. executive_advisory_brief.txt")

Files successfully written to disk:
1. research_portfolio_gl_data.csv
2. division_summary_report.csv
3. executive_advisory_brief.txt


### Summary of Model Capabilities & Strategic Takeaways

This financial planning engine delivers an integrated decision-support framework across three core dimensions:
1. **Grant Accounting & Revenue Modeling:** Simulates multi-year milestone-based revenue recognition and portfolio performance across distinct funding mechanisms.
2. **Labor Recovery & Capacity Planning:** Decomposes scientific labor variances into hourly wage inflation and billable efficiency utilization.
3. **Automated Operational Advisory:** Programmatically generates executive variance waterfalls and plain-language advisory briefings for non-financial research leaders.